In [ ]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"]="1"

In [ ]:
torch.cuda.is_available()

In [ ]:
#os.chdir("/workspace/SamK/prithvi_v2/prithvi-usecases")
os.chdir("/mnt/c/Users/samkh/OneDrive/third_paper/prithviV2/prithvi-usecases")

In [ ]:
import torch
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader
from src.custom_dataset import AquacultureData
from src.models import models
from src.utils import *
import numpy as np
import yaml
import torch.nn as nn
import glob
import os
#import wandb
import argparse
from PIL import Image
import random
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim.lr_scheduler import LambdaLR
from torch.optim import Adam
import tqdm
import pandas as pd

In [ ]:
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

device=config["device_name"]
n_channel=config["model"]["n_channel"]
n_class=config["model"]["n_class"]
n_frame=config["data"]["n_frame"]
n_time_steps=config["data"]["n_time_steps"]
n_iteration=config["n_iteration"]
embed_size=config["model"]["encoder_embed_dim"]
dec_embed_size=config["model"]["dec_embed_dim"]
data_dir=config["data"]["data_dir"]
dataset_name=config["data"]["dataset_name"]
train_csv_path=config["data"]["train_csv_path"]
val_csv_path=config["data"]["val_csv_path"]             
train_batch_size=config["training"]["train_batch_size"]
val_batch_size=config["validation"]["val_batch_size"]
apply_normalization=config["data"]["apply_normalization"]
global_stats=config["data"]["global_stats"]
transformations=config["data"]["transformations"]
learning_rate=config["training"]["learning_rate"]
class_weights=config["class_weights"]
ignore_index=config["ignore_index"]
#segment_input=config["segment_input_path"]
output_dir=config["output_dir"]
#class_weights=config["class_weights"]
#ignore_index=config["ignore_index"]
input_size=config["data"]["input_size"]
patch_size=config["data"]["patch_size"]
checkpoint = os.path.join(output_dir, 'best_checkpoint.pt')
#subset = config["training"]["subset"]
#target_norm = config["data"]["target_norm"]
#input_norm = config["data"]["input_norm"]
#input = config["data"]["input"]
arch = config["model"]["arch"]


# Print all the configuration parameters
print(f"Learning Rate: {learning_rate}")
print(f"Batch Size: {train_batch_size}")
print(f"Number of Epochs: {n_iteration}")
print(f"Number of Input Channel: {n_channel}")
print(f"Number of Segmentation Class: {n_class}")
print(f"Used device name: {device}")
print(f"Checkpoint Path: {checkpoint}")
print(f"Data input dir:{data_dir}")
    
os.makedirs(output_dir, exist_ok=True)

with open(os.path.join(output_dir, 'config.yaml'), 'w') as file:
    yaml.safe_dump(config, file)

In [ ]:
aquaculture_dataset_train = AquacultureData(data_dir, usage="train", dataset_name=dataset_name,
                                            csv_path=train_csv_path, apply_normalization=apply_normalization, 
                                            global_stats=global_stats, trans=transformations)
aquaculture_dataset_val = AquacultureData(data_dir, usage="validation", dataset_name=dataset_name, 
                                          csv_path=val_csv_path, apply_normalization=apply_normalization, 
                                          global_stats=global_stats, trans=transformations)

In [ ]:
# Function to check NaN values in dataset
def check_nan_in_dataset(dataset):
    contains_nan = False
    for i in range(len(dataset)):
        img_chip, label_chip = dataset[i]
        if torch.isnan(img_chip).any() or torch.isnan(label_chip).any():
            print(f"NaN found in the dataset at index {i}")
            contains_nan = True
    if not contains_nan:
        print(f"No NaNs found in the dataset.")

check_nan_in_dataset(aquaculture_dataset_val)

In [ ]:
train_dataloader=DataLoader(aquaculture_dataset_train, batch_size=train_batch_size,
                            shuffle=config["training"]["shuffle"], num_workers=1)
val_dataloader=DataLoader(aquaculture_dataset_val, batch_size=val_batch_size,
                          shuffle=config["validation"]["shuffle"], num_workers=1)

In [ ]:
model_weights = config["prithvi_model_new_weight"] 
    
model_wrapper = models[arch]
#wrapper of prithvi #initialization of prithvi is done by initializing prithvi_loader.py
model=model_wrapper(n_channel, n_class, n_frame, embed_size, input_size, patch_size, model_weights) 
model=model.to(device)

model_dict2 = model.state_dict()
for key in model_dict2.keys():
    print(key)

In [ ]:
optimizer = Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.999), weight_decay=0.05)
optimizer_config = {'grad_clip': None}
scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

In [ ]:
model.train()

for j, (input, target) in enumerate(train_dataloader):
        
    input = input.to(device)
    target = target.to(device)

    optimizer.zero_grad()
    out = model(input)
    loss=segmentation_loss(target, out, device, class_weights, ignore_index)
    loss_i += loss.item() * input.size(0)
    batch_acc = compute_accuracy(target, out)
    acc_dataset_train.append(batch_acc)
    miou_batch=calculate_miou(out, target, device)
    miou_train.append(miou_batch)

    loss.backward()
    optimizer.step()
    scheduler.step()
    
    inner_pbar.update(1)
    inner_pbar.set_description(f"Training Batch Loss: {loss.item()}, Training Batch mIoU: {miou_batch}", refresh=True)

In [ ]:
from src.models.Prithvi import TemporalViTEncoder
model_weights = config["prithvi_model_new_weight"] 
model = TemporalViTEncoder(
    img_size=input_size,      # Set correct values based on your architecture
    patch_size=patch_size, 
    num_frames=1, 
    tubelet_size=1, 
    in_chans=6, 
    embed_dim=embed_size, 
    depth=24, 
    num_heads=16, 
    mlp_ratio=4, 
    norm_layer=nn.LayerNorm, 
    norm_pix_loss=False, 
    pretrained=None  # We will load weights separately
)

in_params = torch.load(model_weights, map_location="cpu")
model_dict = model.state_dict()

In [ ]:
for key in model_dict.keys():
    print(key)

In [ ]:
for key in in_params.keys():
    print(key)

In [ ]:
chpt_path2 = "/workspace/SamK/prithvi_v2/results/100p/best_checkpoint.pt"
in_params2 = torch.load(chpt_path2, map_location="cpu")

for key in in_params2["model_state_dict"].keys():
    print(key)

In [ ]:
in_params['encoder.patch_embed.proj.weight']

In [ ]:
in_params2['model']['patch_embed.proj.weight']

In [ ]:
for key in ['encoder.pos_embed', 'decoder.decoder_pos_embed']:
    if key in in_params:
        print(key)

In [ ]:
chpt_path = "/workspace/SamK/prithvi_v2/results/100p/best_checkpoint.pt"
in_params2 = torch.load(chpt_path, map_location="cpu")

for key in in_params2["model_state_dict"].keys():
    print(key)

In [ ]:
import torch
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Allocated memory: {torch.cuda.memory_allocated(i) / 1024**2:.2f} MB")
    print(f"  Reserved memory: {torch.cuda.memory_reserved(i) / 1024**2:.2f} MB")

In [ ]:
from pathlib import Path
img_fname = Path("/workspace/SamK/prithvi_v2/aquaculture_data/ood_pred_dataset/images/mexico_034042_2022_id_1_band_chip.tif")
img_fname = Path("/workspace/SamK/prithvi_v2/aquaculture_data/ood_pred_dataset/images/bangladesh_khulna_138044_2018_id_153_band_chip.tif")

img_id = '_'.join(img_fname.stem.split('_')[:-2])

In [ ]:
img_id